# P12 — Estimation du montant des marchés publics (DECP)

**Contexte métier.** Journalistes (data-investigation), acheteurs publics, fédérations d'entreprises : estimer le montant d'un marché à partir de ses caractéristiques (procédure, durée, CPV, acheteur) pour repérer les anomalies — marchés surdimensionnés, grappe d'acheteurs. Enjeu : 100 Md€ de commande publique/an, transparence.

**Objectif de l'EDA.** Identifier la cible (montant en €, très asymétrique → log) et les régresseurs (procédure, forme de prix, durée, CPV, nombre d'offres).

**Données.** Données essentielles de la commande publique — DECP consolidé 2022 (data.gouv.fr). Licence ouverte Etalab 2.0. 8 756 marchés avec montant.

## 1. Téléchargement

Fichier consolidé DECP 2022 (JSON) :

In [ ]:
import urllib.request, os
os.makedirs("data", exist_ok=True)

URL = "https://static.data.gouv.fr/resources/donnees-essentielles-de-la-commande-publique-fichiers-consolides/20241114-000009/decp-2022.json"
if not os.path.exists("data/decp-2022.json"):
    urllib.request.urlretrieve(URL, "data/decp-2022.json")
print(os.path.getsize("data/decp-2022.json") / 1e6, "Mo")

## 2. Chargement

Le JSON contient une clé `marches` (liste). On aplatit les champs utiles, dont le département de l'acheteur via les 2 premiers chiffres du SIRET.

In [ ]:
import json
import pandas as pd, numpy as np

d = json.load(open("data/decp-2022.json"))
rows = []
for m in (d["marches"] if isinstance(d, dict) else d):
    ach = m.get("acheteur") or {}
    rows.append({
        "montant": m.get("montant"), "procedure": m.get("procedure"),
        "forme_prix": m.get("formePrix"), "duree_mois": m.get("dureeMois"),
        "cpv2": (m.get("codeCPV") or "")[:2], "nature": m.get("nature"),
        "offres": m.get("offresRecues"), "avance": m.get("attributionAvance"),
        "sous_traitance": m.get("sousTraitanceDeclaree"), "innovant": m.get("marcheInnovant"),
        "acheteur_siret": ach.get("id"), "n_titulaires": len(m.get("titulaires") or []),
        "date_notif": m.get("dateNotification"), "dept": str(ach.get("id"))[:2],
    })
df = pd.DataFrame(rows)
for c in ["montant", "duree_mois", "offres", "n_titulaires"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["log_montant"] = np.log10(df["montant"].clip(lower=1))
print(len(df), "marchés")

## 3. Analyse

Distribution du montant (et du log), médiane par procédure, par forme de prix, par division CPV, par durée ; effet du nombre d'offres reçues.

In [ ]:
dfa = df.dropna(subset=["montant"])
print("Montant (M€) :", (dfa["montant"].describe(percentiles=[.25, .5, .75, .9]) / 1e6).round(3).to_dict())
print()
print("Médiane (€) par procédure :")
print(dfa.groupby("procedure")["montant"].median().round(0).sort_values(ascending=False).to_dict())
print()
print("Médiane (€) par forme de prix :", dfa.groupby("forme_prix")["montant"].median().round(0).to_dict())
print("Médiane (€) par division CPV (2 chiffres) :")
print(dfa.groupby("cpv2")["montant"].median().round(0).sort_values(ascending=False).head(6).to_dict())
print()
c = dfa[["log_montant", "duree_mois", "offres", "n_titulaires"]].corr("spearman")["log_montant"]
print("Corr. Spearman avec log(montant) :", c.round(2).to_dict())

## 4. Résultats observés (exécution du 16/10/2026)

- n marches : 8756
- n avec montant : 8756
- montant stats MEUR : count : 0.01, mean : 0.97, std : 25.12, min : 0.00, 10% : 0.01, 25% : 0.03, 50% : 0.09, 75% : 0.25, 90% : 0.74, max :…
- log10 montant stats : count : 8 756.00, mean : 4.94, std : 0.82, min : 0.00, 10% : 4.02, 50% : 4.95, 90% : 5.87, max : 9.31
- montant median par procedure : Dialogue compétitif : 13 834 640.00, Procédure avec négociation : 632 205.00, Appel d'offres restreint : 370 000.00, …
- montant median par forme prix :  : 0.00, Forfaitaire : 86 000.00, Mixte : 169 540.00, Unitaire : 82 980.00
- montant median par cpv2 : 65 : 800 000.00, 55 : 320 000.00, 14 : 250 000.00, 41 : 196 253.00, 72 : 176 200.00, 19 : 170 000.00, 60 : 145 548.00…
- montant median par duree : (-0.001, 3.0] : 48 783.00, (3.0, 6.0] : 61 405.00, (6.0, 12.0] : 70 454.00, (12.0, 24.0] : 115 684.00, (24.0, 36.0] :…
- procedure dist : Procédure adaptée : 6050, Appel d'offres ouvert : 2280, Marché passé sans publicité ni mise en concurrence préalable …
- n acheteurs : 595
- offres stats : count : 8 756.00, mean : 4.93, std : 143.84, min : 0.00, 50% : 3.00, 90% : 6.00, max : 13 454.00
- montant median par n offres : 0 : 47 000.00, 1 : 74 765.00, 2 : 77 741.00, 3 : 95 613.00, 4 : 99 529.00, 5 : 120 000.00
- pct avance : 9.40
- pct innovant : 2.80
- dept top10 n marches : 21 : 2364, 20 : 1003, 45 : 570, 13 : 528, 27 : 461, 24 : 428, 44 : 258, 94 : 240, 25 : 219, 78 : 185
- corr spearman log montant : log_montant : 1.00, duree_mois : 0.20, offres : 0.13, n_titulaires : 0.08
- missing pct : montant : 0.00, procedure : 0.00, forme_prix : 0.00, duree_mois : 0.00, cpv2 : 0.00, nature : 0.00, offres : 0.00, av…

## 5. Variable cible et régresseurs

**Cible (régression) : `montant` en € — à modéliser en log10** (médiane 90 k€, max 2 Md€ : huit décades de dynamique).

**Régresseurs retenus :**
- `procedure` : effet majeur (médiane 13,8 M€ pour un dialogue compétitif, 79 k€ pour une procédure adaptée)
- `cpv2` (division nomenclature CPV) : 800 k€ pour les services financiers, 119 k€ pour la construction
- `duree_mois` (corr. 0,20 avec le log-montant), `forme_prix` (forfaitaire vs unitaire)
- `offres` reçues (corr. 0,13) et `dept` de l'acheteur
- `n_titulaires`, `nature` du marché, `ccag`

**Pièges :**
- Montants extrêmes (2 Md€) : travailler en log, évaluer en MAE relative, winsoriser pour la visualisation
- Les DECP 2022 sont une base remaniée a posteriori (les marchés sont republiés quand ils évoluent) : un marché peut apparaître plusieurs fois — dédupliquer sur `id`
- 1,6 % de montants nuls ou manquants à écarter
- Le montant est structuré par le CPV : sans cette feature le modèle reste vague ; avec, l'essentiel du signal vient de la nomenclature